# Brain Tumor Classifier using KMeans

- classifies MRI scans as glioma, meningioma, pituitary, or no tumor   
- uses PCA to reduce dimensionality of the data images ot be more manageable
- trains on 256x256 grayscale images from images/Training and images/Testing  
- tracks training and validation accuracy each epoch  
- includes a quick test function to predict any single image and show its confidence


In [14]:
import os
import numpy as np
import pandas as pd
import cv2
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, confusion_matrix

# load data into two arrays of data and labels
def load_data(root, classes):
    data = []
    labels = []
    for l in classes:
        folder = os.path.join(root, l)
        for d in tqdm(os.listdir(folder), desc=f"Loading {l}"):
            path = os.path.join(folder, d)
            if d.endswith(".npy"):
                data.append(np.load(path).flatten())
                labels.append(l)
    return np.array(data), np.array(labels)

In [21]:
root = os.path.dirname(os.path.abspath("kmeans.ipynb"))
trainingFolder = os.path.join(root, "dataset\Training")
classes = ["glioma", "meningioma", "pituitary", "notumor"]
xTrain, yTrain = load_data(trainingFolder, classes)
# dimensionality reduction
pca = PCA(n_components=100)
xTrainPCA = pca.fit_transform(xTrain)
kmeans = KMeans(n_clusters=4, random_state=42)
# fit kmeans
trainingClusters = kmeans.fit_predict(xTrainPCA)
# output adjusted rand score
print("ARI: ", adjusted_rand_score(yTrain, trainingClusters))
df = pd.DataFrame({"true": yTrain, "cluster": trainingClusters})
print(df.groupby(["cluster", "true"]).size())

Loading notumor: 100%|██████████| 3190/3190 [00:00<00:00, 7044.38it/s]


ARI:  0.142104273703949
cluster  true      
0        glioma         148
         meningioma     404
         notumor       1259
         pituitary      335
1        glioma         344
         meningioma     265
         notumor         27
         pituitary      222
2        glioma         385
         meningioma     314
         notumor         39
         pituitary      502
3        glioma         444
         meningioma     356
         notumor        270
         pituitary      398
dtype: int64
